# Lab 1 at home · Two tables of your own in DuckDB

**Due:** Sunday, October 25 at 11:59 PM · **Submit:** your repo URL on CourseWorks

In class you loaded one table (penguins), found a problem and fixed it. Now do the same with **two related tables you choose**. In Session 2 we'll join them, so pick two that can be connected.

You will:
1. Pick two related public datasets and upload them
2. Load both into DuckDB and check what DuckDB decided about each column
3. Find the column the two tables share (the **key**) and check how well they match
4. Write five SQL queries that help explain your data
5. Write observations and a dataset card for each table

## 1. Pick two related tables

**Related** means they share a column you could match rows on: an ID, a name, a ZIP code, a country code, a date.

**Example pairs**
- **World Bank:** any indicator download (e.g. GDP per country per year) comes with a second file of country details (region, income group). The shared column is the country code.
- **Our World in Data:** two charts for the same countries, e.g. life expectancy and health spending. Shared columns: country and year.
- **NYC Open Data:** 311 complaints and population by ZIP code, or the street tree census and neighborhood boundaries. Shared column: ZIP code or neighborhood.
- **Your own idea:** any two public tables that share a column.

**Rules**
- **Public, open data only.** Your repo is public, and later this data goes to a free AI tier. Nothing from work, nothing personal, nothing behind a login.
- **CSV format.** If a file is Excel, open it and save it as CSV first.
- **Each file under 100 MB.** Between 1,000 and 1,000,000 rows is a good size for the main table; the second can be small.
- **Reuse allowed.** Check each license or terms page, and note it in your dataset card.

**Places to look:** [NYC Open Data](https://opendata.cityofnewyork.us) · [data.gov](https://data.gov) · [World Bank Open Data](https://data.worldbank.org) · [Our World in Data](https://ourworldindata.org) · [Kaggle Datasets](https://www.kaggle.com/datasets) (check each license)

## 2. Upload them

1. Download both CSVs to your computer.
2. In this Codespace, drag both files into the **`data`** folder in the file list on the left.
3. Rename each to something short with no spaces, e.g. `gdp.csv` and `countries.csv` (right-click → **Rename**).
4. Fill in the cell below and run it. Table names should be short, lowercase, no spaces: you'll type them in your SQL.

In [1]:
FILE_A  = "pluto.csv"     # <- your main table's file name
TABLE_A = "pluto"                   # <- a short name for it, e.g. "gdp"

FILE_B  = "zap.csv"   # <- the related table's file name
TABLE_B = "zap"                 # <- a short name for it, e.g. "countries"

In [2]:
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_A = ROOT / "data" / FILE_A
DATA_B = ROOT / "data" / FILE_B

missing = [p.name for p in (DATA_A, DATA_B) if not p.exists()]
if missing:
    print("Can't find:", ", ".join(missing))
    print("Files in data/ right now:")
    for f in sorted((ROOT / "data").iterdir()):
        print("  ", f.name)
else:
    con = duckdb.connect(str(ROOT / "lab1_home.duckdb"))
    for p in (DATA_A, DATA_B):
        print(f"Found data/{p.name} ({p.stat().st_size / 1_000_000:.1f} MB)")
    print("Connected.")

Found data/pluto.csv (13.5 MB)
Found data/zap.csv (2.7 MB)
Connected.


## 3. First look

Preview the first 10 rows of each raw file before loading.

In [3]:
con.sql(f"SELECT * FROM read_csv('{DATA_A}') LIMIT 10")

┌──────────────┬─────────┬───────┬───────┬──────────────────────┬───────┬─────────┬───────────┬───────────┬───────────┬───────────┬──────────┬──────────┬─────────┬─────────┬─────────┬───────────┬───────────┬───────────┬─────────┬─────────┬──────────┬─────────┬─────────┬──────────┬────────────┬───────────┬───────────┬────────────┬────────────┬──────────┬──────────┬─────────┬──────────┬───────────┬────────┬─────────────────────┬────────────────────────────────────┬────────────┬─────────────┬─────────┐
│     bbl      │ borough │ block │  lot  │       address        │  cd   │ bct2020 │ zonedist1 │ zonedist2 │ zonedist3 │ zonedist4 │ overlay1 │ overlay2 │ spdist1 │ spdist2 │ spdist3 │ ltdheight │ splitzone │ bldgclass │ landuse │ lotarea │ bldgarea │ comarea │ resarea │ unitsres │ unitstotal │ numfloors │ yearbuilt │ yearalter1 │ yearalter2 │ builtfar │ residfar │ commfar │ facilfar │ affresfar │ mnffar │      landmark       │              histdist              │  latitude  │  longitude  │ 

In [4]:
con.sql(f"SELECT * FROM read_csv('{DATA_B}') LIMIT 10")

┌────────────┬────────────┬───────────────────┬─────────────────┬───────────────┬───────────┬─────────────────────┬────────────────────┬──────────────────┬────────────────┐
│ project_id │    bbl     │ validated_borough │ validated_block │ validated_lot │ validated │   validated_date    │ unverified_borough │ unverified_block │ unverified_lot │
│  varchar   │   int64    │      varchar      │      int64      │     int64     │  boolean  │      timestamp      │      varchar       │      int64       │     int64      │
├────────────┼────────────┼───────────────────┼─────────────────┼───────────────┼───────────┼─────────────────────┼────────────────────┼──────────────────┼────────────────┤
│ P2012M0178 │ 1000000000 │ Manhattan         │               0 │             0 │ NULL      │ 2018-04-10 04:00:00 │ Manhattan          │             NULL │           NULL │
│ P2012M0455 │ 1000000000 │ Manhattan         │               0 │             0 │ NULL      │ 2018-04-10 04:00:00 │ Manhattan          

## 4. Load both and check the types

Load each file into a table, then `DESCRIBE` it. Compare each column's type with what it *should* be. A number or date column stored as `VARCHAR` (text) is the same kind of problem you fixed in class.

In [5]:
for table, path in ((TABLE_A, DATA_A), (TABLE_B, DATA_B)):
    con.sql(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM read_csv('{path}')")
    rows = con.sql(f"SELECT count(*) FROM {table}").fetchone()[0]
    print(f"{table}: {rows:,} rows loaded")

pluto: 42,504 rows loaded
zap: 27,011 rows loaded


In [6]:
con.sql(f"DESCRIBE {TABLE_A}")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ bbl         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ borough     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ block       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ lot         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ address     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ cd          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ bct2020     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ zonedist1   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ zonedist2   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ zonedist3   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│    ·        │   · 

In [7]:
con.sql(f"DESCRIBE {TABLE_B}")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ project_id         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ bbl                │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ validated_borough  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ validated_block    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ validated_lot      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ validated          │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ validated_date     │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ unverified_borough │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ unverified_block   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

`SUMMARIZE` profiles every column at once: min, max, how many distinct values, and `null_percentage` (how much is missing).

In [8]:
con.sql(f"SUMMARIZE {TABLE_A}")

┌─────────────┬─────────────┬───────────────────────────────────────────────────────┬─────────────────────────────────────────────────┬───────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬────────────────────┬───────┬─────────────────┐
│ column_name │ column_type │                          min                          │                       max                       │ approx_unique │        avg         │         std         │        q25         │        q50         │        q75         │ count │ null_percentage │
│   varchar   │   varchar   │                        varchar                        │                     varchar                     │     int64     │      varchar       │       varchar       │      varchar       │      varchar       │      varchar       │ int64 │  decimal(9,2)   │
├─────────────┼─────────────┼───────────────────────────────────────────────────────┼─────────────────────────────────────────────────┼─────────────

In [9]:
con.sql(f"SUMMARIZE {TABLE_B}")

┌────────────────────┬─────────────┬─────────────────────┬─────────────────────┬───────────────┬────────────────────────────┬────────────────────┬─────────────────────┬─────────────────────┬────────────────────────────┬───────┬─────────────────┐
│    column_name     │ column_type │         min         │         max         │ approx_unique │            avg             │        std         │         q25         │         q50         │            q75             │ count │ null_percentage │
│      varchar       │   varchar   │       varchar       │       varchar       │     int64     │          varchar           │      varchar       │       varchar       │       varchar       │          varchar           │ int64 │  decimal(9,2)   │
├────────────────────┼─────────────┼─────────────────────┼─────────────────────┼───────────────┼────────────────────────────┼────────────────────┼─────────────────────┼─────────────────────┼────────────────────────────┼───────┼─────────────────┤
│ project_id    

## 5. Fix what loaded wrong (if anything)

If every type looks right, write "No fixes needed" in the text cell after the next one and skip ahead.

Otherwise, open the CSV (double-click it in the `data` folder) and find **how missing values are written** in the columns that came in as text. Edit the `nullstr` list below to match, and run it for the table that needs fixing.

In [ ]:
# Change TABLE_A / DATA_A to TABLE_B / DATA_B if it's the second table that needs fixing
con.sql(f"""
    CREATE OR REPLACE TABLE {TABLE_A} AS
    SELECT * FROM read_csv('{DATA_A}', nullstr = ['NA', 'N/A', ''])
""")
con.sql(f"DESCRIBE {TABLE_A}")

**What did you fix, and how did you know?** (Double-click to edit.)



## 6. Find the key: how do the two tables connect?

The **key** is the column both tables share. Type its name in each table below. It may be spelled differently in each, e.g. `Country Code` in one and `iso3` in the other.

The next cell counts the distinct key values in each table and how many of them appear in both. (It treats both as text, so a ZIP code stored as a number still matches one stored as text.)

In [10]:
KEY_A = "bbl"   # <- the shared column's name in your main table
KEY_B = "bbl"   # <- its name in the second table

In [ ]:
con.sql(f"""
    WITH a AS (SELECT DISTINCT CAST(CAST("{KEY_A}" AS BIGINT)AS VARCHAR) AS k FROM {TABLE_A} WHERE bbl IS NOT NULL),
         b AS (SELECT DISTINCT CAST("{KEY_B}" AS VARCHAR) AS k FROM {TABLE_B} WHERE bbl IS NOT NULL)  
         
    SELECT
        (SELECT count(*) FROM a)                           AS keys_in_a,
        (SELECT count(*) FROM b)                           AS keys_in_b,
        (SELECT count(*) FROM a WHERE k IN (SELECT k FROM b)) AS keys_in_both
""")

┌───────────┬───────────┬──────────────┐
│ keys_in_a │ keys_in_b │ keys_in_both │
│   int64   │   int64   │    int64     │
├───────────┼───────────┼──────────────┤
│     42504 │     15485 │        10971 │
└───────────┴───────────┴──────────────┘

In [21]:
con.sql("""
SELECT DISTINCT bbl
FROM zap
WHERE bbl IS NOT NULL
  AND bbl NOT IN (
      SELECT CAST(bbl AS BIGINT)
      FROM pluto
      WHERE bbl IS NOT NULL
  )
ORDER BY bbl
LIMIT 10
""")

┌────────────┐
│    bbl     │
│   int64    │
├────────────┤
│ 1000000000 │
│ 1000010000 │
│ 1000017501 │
│ 1000030006 │
│ 1000057510 │
│ 1000060006 │
│ 1000060013 │
│ 1000070036 │
│ 1000080000 │
│ 1000080032 │
└────────────┘
   10 rows  

**Describe the relationship** (double-click to edit):

- **Key column:** Both tables use `bbl` as the shared key. I standardized the BBL format and excluded NULL keys when comparing them.
- **How they relate:** PLUTO contains one row per BBL in this extract, while ZAP can contain multiple records for the same BBL because a parcel may be associated with multiple projects. The relationship is one-to-many from PLUTO to ZAP.
- **How well they match:** The tables share 10,971 distinct non-null BBLs. There are 31,533 BBLs only in PLUTO and 4,514 only in ZAP. For example, BBL `1000000000` appears in ZAP but not in the current PLUTO extract. Its block and lot components are both zero, suggesting that it may be a placeholder or an incomplete identifier.  



## 7. Five queries that explain your data

Write one query of each kind. Under each, double-click the text cell and write one line: **what does this result tell us?** Use either table.

Tip: if a column name has spaces or capitals, wrap it in double quotes: `"Country Name"`. Start from a query in the in-class notebook and change one thing.

### Query 1 · Size and shape
*How many rows, and over what time period or area?* Useful SQL: `count(*), min(), max()`

In [22]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT bbl) AS unique_parcels,
    MIN(borough) AS borough_min,
    MAX(borough) AS borough_max
FROM pluto
""")

┌────────────┬────────────────┬─────────────┬─────────────┐
│ total_rows │ unique_parcels │ borough_min │ borough_max │
│   int64    │     int64      │   varchar   │   varchar   │
├────────────┼────────────────┼─────────────┼─────────────┤
│      42504 │          42504 │ MN          │ MN          │
└────────────┴────────────────┴─────────────┴─────────────┘

**What this tells us:** The PLUTO extract contains 42,504 records, each with a unique BBL. Both the minimum and maximum borough values are “MN,” indicating that all non-null borough values refer to Manhattan.

### Query 2 · Filter
*Which rows meet a condition you care about?* Useful SQL: `WHERE`

In [23]:
con.sql("""
SELECT
    CAST(bbl AS BIGINT) AS bbl,
    address,
    lotarea,
    builtfar,
    zonedist1
FROM pluto
WHERE lotarea >= 5000
  AND builtfar < 1
  AND builtfar >= 0
ORDER BY bbl
LIMIT 10
""")

┌────────────┬──────────────────────┬─────────┬──────────┬───────────┐
│    bbl     │       address        │ lotarea │ builtfar │ zonedist1 │
│   int64    │       varchar        │  int64  │  double  │  varchar  │
├────────────┼──────────────────────┼─────────┼──────────┼───────────┤
│ 1000010010 │ 140 CARDER ROAD      │ 7577714 │     0.33 │ R3-2      │
│ 1000010100 │ CARDER ROAD          │   23121 │     0.43 │ R3-2      │
│ 1000010111 │ ANDES ROAD           │   64451 │     0.99 │ R3-2      │
│ 1000010150 │ COMFORT ROAD         │   64181 │     0.35 │ R3-2      │
│ 1000010201 │ 1 ELLIS ISLAND       │ 2764190 │     0.22 │ NULL      │
│ 1000010301 │ JOE DIMAGGIO HIGHWAY │  241719 │      0.0 │ NULL      │
│ 1000010401 │ JOE DIMAGGIO HIGHWAY │  246896 │      0.0 │ NULL      │
│ 1000020002 │ 10 SOUTH STREET      │  191502 │     0.82 │ C4-6      │
│ 1000020003 │ MARGINAL STREET      │   38800 │     0.19 │ C4-6      │
│ 1000020023 │ 1 PIER 6             │  510025 │     0.05 │ C4-6      │
└─────

**What this tells us:** These 10 parcels have relatively large lot areas but low building density, which miss their air right.They may be potential redevelopment opportunities. However, this does not necessarily mean they can be redeveloped; zoning and other site restrictions need to be checked first.

### Query 3 · Group
*How does a count or average differ across categories?* Useful SQL: `GROUP BY`

In [ ]:
con.sql("""
    -- write your query here; use your own table names
    SELECT * FROM table_a LIMIT 5
""")

**What this tells us:** 

### Query 4 · Rank
*What are the top or bottom 10 of something?* Useful SQL: `ORDER BY ... LIMIT 10`

In [ ]:
con.sql("""
    -- write your query here; use your own table names
    SELECT * FROM table_a LIMIT 5
""")

**What this tells us:** 

### Query 5 · Quality check
*What's missing, duplicated or impossible?* Useful SQL: `count(*) - count(column), count(DISTINCT ...)`

In [ ]:
con.sql("""
    -- write your query here; use your own table names
    SELECT * FROM table_a LIMIT 5
""")

**What this tells us:** 

## 8. Three observations

Specific things you noticed that you'd want to check before trusting this data. Point to numbers.

1. 
2. 
3. 

**One question you'd want a non-technical person to be able to ask of this data:**



## 9. Dataset cards

| | Main table | Second table |
|---|---|---|
| **Dataset name** | | |
| **Source (link)** | | |
| **Who collected it, and why** | | |
| **License or terms of use** | | |
| **Rows × columns** | | |
| **Time period covered** | | |
| **How often it's updated** | | |

## 10. Submit

1. Run the cell below to close the connection.
2. Fill in the lifecycle map of your data in `labs/lab1.md`.
3. **Save this notebook (Cmd+S)**, then commit and push: **Source Control** → message `Lab 1` → **Commit** → **Sync Changes**.
4. Check on GitHub that both CSVs, both notebooks and `labs/lab1.md` are there, with results visible.
5. Paste your repo URL into the Lab 1 assignment on CourseWorks.

In [ ]:
con.close()
print("Done. Save, commit, push, and submit your repo URL.")